In [1]:
import os

from dotenv import load_dotenv

from langchain_aws import ChatBedrock
from langchain_openai import ChatOpenAI

In [2]:
load_dotenv()

True

# Connect to Bedrock...

In [3]:
# llm.py
llm = ChatBedrock(
    model_id=os.getenv(
        "BEDROCK_CHAT_MODEL_ID"
    ),
    region_name=os.getenv(
        "AWS_REGION"
    ),
    model_kwargs={
        "temperature": 0.2
    }
)

# Connect to OpenAI

In [ ]:
llm = ChatOpenAI(
    model=os.getenv(
        "OPENAI_CHAT_MODEL"
    ),
    temperature=0.2
)

# Load Document

In [4]:
# ingest.py
from langchain_community.document_loaders import (
    DirectoryLoader,
    TextLoader
)

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

from langchain_aws import BedrockEmbeddings
from langchain_openai import OpenAIEmbeddings

from langchain_pinecone import PineconeVectorStore

from pinecone import Pinecone

C:\Users\m\AppData\Local\Temp\ipykernel_1368\3328892794.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (


In [5]:
loader = DirectoryLoader(
    "knowledge-base",
    glob="**/*.txt",
    loader_cls=TextLoader
)

documents = loader.load()


print(
    f"Loaded {len(documents)} documents"
)

Loaded 6 documents


In [6]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)


chunks = splitter.split_documents(
    documents
)


print(
    f"Created {len(chunks)} chunks"
)

Created 9 chunks


# Bedrock Embeddings

In [7]:
embeddings = BedrockEmbeddings( 
    model_id=os.getenv(
        "BEDROCK_EMBEDDING_MODEL_ID"
    ),
    region_name=os.getenv(
        "AWS_REGION"
    )
)

# OpenAI Embeddings

In [ ]:
embeddings = OpenAIEmbeddings(
    model=os.getenv(
        "OPENAI_EMBEDDING_MODEL"
    )
)

# Upload Vectors to Pinecone

In [8]:
pc = Pinecone(
    api_key=os.getenv(
        "PINECONE_API_KEY"
    )
)

# Indexing - Bedrock

In [9]:
index = pc.Index(
    os.getenv(
        "PINECONE_INDEX_NAME_BEDROCK"
    )
)

# Indexing - OpenAI

In [ ]:
index = pc.Index(
    os.getenv(
        "PINECONE_INDEX_NAME_OPENAI"
    )
)

In [10]:
vectorstore = PineconeVectorStore(
    index=index,
    embedding=embeddings
)


vectorstore.add_documents(
    chunks
)


print(
    "Knowledge base loaded into Pinecone"
)

Knowledge base loaded into Pinecone


# Do a Similarity Search

In [11]:
results = vectorstore.similarity_search(
    "Does my Gold plan cover physiotherapy?",
    k=3
)

for doc in results:
    print(doc.page_content)

4. Diagnostic Imaging & Physiotherapy
Basic X-rays are covered in-office. MRIs and CT scans require prior authorization (refer to 04_Prior_Authorization_Guide.pdf). Physiotherapy is capped at 20 visits/year for Gold plans.

5. Prescription Drugs & Limits
Prescriptions are tiered: Generic (Tier 1), Preferred Brand (Tier 2), Non-Preferred (Tier 3), and Specialty (Tier 4). Specialty drugs require prior approval.
4. Diagnostic Imaging & Physiotherapy
Basic X-rays are covered in-office. MRIs and CT scans require prior authorization (refer to 04_Prior_Authorization_Guide.pdf). Physiotherapy is capped at 20 visits/year for Gold plans.

5. Prescription Drugs & Limits
Prescriptions are tiered: Generic (Tier 1), Preferred Brand (Tier 2), Non-Preferred (Tier 3), and Specialty (Tier 4). Specialty drugs require prior approval.
HealthSecure Insurance - Benefits Guide

1. Plan Overview (Bronze, Silver, Gold)
HealthSecure offers three core tiers:
â€¢ Bronze: Lower premiums, higher deductibles ($5,000 

# Similarity Search with Scores

In [12]:
results = vectorstore.similarity_search_with_score(
    "Does my Gold plan cover physiotherapy?",
    k=5
)

for doc, score in results:
    print("Score:", score)
    print(doc.page_content)
    print("---")

Score: 0.413295746
4. Diagnostic Imaging & Physiotherapy
Basic X-rays are covered in-office. MRIs and CT scans require prior authorization (refer to 04_Prior_Authorization_Guide.pdf). Physiotherapy is capped at 20 visits/year for Gold plans.

5. Prescription Drugs & Limits
Prescriptions are tiered: Generic (Tier 1), Preferred Brand (Tier 2), Non-Preferred (Tier 3), and Specialty (Tier 4). Specialty drugs require prior approval.
---
Score: 0.413220257
4. Diagnostic Imaging & Physiotherapy
Basic X-rays are covered in-office. MRIs and CT scans require prior authorization (refer to 04_Prior_Authorization_Guide.pdf). Physiotherapy is capped at 20 visits/year for Gold plans.

5. Prescription Drugs & Limits
Prescriptions are tiered: Generic (Tier 1), Preferred Brand (Tier 2), Non-Preferred (Tier 3), and Specialty (Tier 4). Specialty drugs require prior approval.
---
Score: 0.270000458
HealthSecure Insurance - Benefits Guide

1. Plan Overview (Bronze, Silver, Gold)
HealthSecure offers three co

# Create Pinecone Retriever

In [14]:
# rag.py
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k":3
    }
)

# Similarity Search via Invoke

In [15]:
question = "Does my Gold plan cover physiotherapy?"

results = retriever.invoke(question)

for doc in results:
    print("SOURCE:", doc.metadata["source"])
    print(doc.page_content)
    print("--------")

SOURCE: knowledge-base\02_Benefits_Guide.txt
4. Diagnostic Imaging & Physiotherapy
Basic X-rays are covered in-office. MRIs and CT scans require prior authorization (refer to 04_Prior_Authorization_Guide.pdf). Physiotherapy is capped at 20 visits/year for Gold plans.

5. Prescription Drugs & Limits
Prescriptions are tiered: Generic (Tier 1), Preferred Brand (Tier 2), Non-Preferred (Tier 3), and Specialty (Tier 4). Specialty drugs require prior approval.
--------
SOURCE: knowledge-base\02_Benefits_Guide.txt
4. Diagnostic Imaging & Physiotherapy
Basic X-rays are covered in-office. MRIs and CT scans require prior authorization (refer to 04_Prior_Authorization_Guide.pdf). Physiotherapy is capped at 20 visits/year for Gold plans.

5. Prescription Drugs & Limits
Prescriptions are tiered: Generic (Tier 1), Preferred Brand (Tier 2), Non-Preferred (Tier 3), and Specialty (Tier 4). Specialty drugs require prior approval.
--------
SOURCE: knowledge-base\02_Benefits_Guide.txt
HealthSecure Insuranc

# Test RAG with LLM

In [16]:
from langchain_core.prompts import ChatPromptTemplate

question = "What services do you provide?"
docs = retriever.invoke(
    "What is my deductible?"
)

context = "\n\n".join(
    f"Source: {doc.metadata['source']}\n{doc.page_content}"
    for doc in docs
)

In [17]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            You are HealthSecure AI Assistant.
            
            Your responsibilities:
            
            - Answer questions only using the supplied context.
            - If the answer is not contained in the context, say:
              "I couldn't find that information in the HealthSecure knowledge base."
            - Never invent insurance policies.
            - Explain information clearly in simple language.
            - Use bullet points where appropriate.
            - If a user asks for personal information, tell them their member information will be retrieved separately.
            """
        ),
        (
            "human",
            """
            Context:
            
            {context}
            
            Question:
            
            {question}
            """
        )
    ]
)

In [18]:
messages = prompt.invoke({
    "context": context,
    "question": question
})

response = llm.invoke(messages)

print(response.content)

HealthSecure Insurance provides the following services:

- **Primary & Specialist Care**
  - Primary care visits require a $25 copay for Silver and Gold plans.
  - Specialist care requires a primary care referral for Bronze plans.

- **Emergency & Preventive Care**
  - Emergency care is covered 100% after copay at any facility.
  - Preventive care (annual physicals, vaccinations) is covered with $0 copay across all plans.

For more detailed information on network providers and coverage policies, please refer to the 03_Coverage_Policies.pdf document.


In [19]:
# database.py
from sqlalchemy import create_engine
from sqlalchemy import text

load_dotenv()

DATABASE_URL = (
    f"postgresql://"
    f"{os.getenv('POSTGRES_USER')}:"
    f"{os.getenv('POSTGRES_PASSWORD')}@"
    f"{os.getenv('POSTGRES_HOST')}:"
    f"{os.getenv('POSTGRES_PORT')}/"
    f"{os.getenv('POSTGRES_DB')}"
)


engine = create_engine(
    DATABASE_URL
)

In [20]:
# tools.py
from langchain_core.tools import tool

# from app.rag import retriever
# from app.database import engine

@tool("search_knowledge_base")
def search_knowledge_base(
    question:str
):
    """
    Search HealthSecure policy documents.
    """

    docs = retriever.invoke(
        question
    )


    return "\n\n".join(
        [
            d.page_content
            for d in docs
        ]
    )

In [21]:
@tool
def get_member(member_id: int) -> str:
    """
    Retrieve member information using member_id.
    member_id is required and must be an integer.
    """

    if not member_id:
        return "Member ID is required."

    try:
        member_id = int(member_id)
    except ValueError:
        return "Member ID must be a number."

    sql = """
    SELECT *
    FROM members
    WHERE member_id=:id
    """

    with engine.connect() as conn:
        result = conn.execute(
            text(sql),
            {
                "id": member_id
            }
        )

        row = result.fetchone()
    if row:
        return str(dict(row._mapping))

    return "Member not found"

In [22]:
@tool
def get_claim_status(claim_id: str) -> dict:
    """
    Retrieve claim status using a claim ID.
    """

    sql = """
    SELECT *
    FROM claims
    WHERE claim_id = :id
    """

    with engine.connect() as conn:
        result = conn.execute(
            text(sql),
            {
                "id": claim_id
            }
        )

        row = result.fetchone()

    if not row:
        return {
            "message": "Claim not found"
        }

    return dict(row._mapping)

In [23]:
# agent.py
from langchain.agents import create_agent

from langchain_core.prompts import ChatPromptTemplate

# from app.llm import llm
"""from app.tools import (
    search_knowledge_base,
    get_member,
    get_claim_status
)"""

'from app.tools import (\n    search_knowledge_base,\n    get_member,\n    get_claim_status\n)'

In [24]:
tools=[
    search_knowledge_base,
    get_member,
    get_claim_status
]

system_prompt = """
You are HealthSecure AI Assistant.

Your role:
Help HealthSecure members and non-members.

Rules:

1. Use tools whenever required.

2. Never invent information.

3. If required information is missing, ask the user.

4. Use:
- search_knowledge_base for insurance policies and benefits.
- get_member for member-specific information.
- get_claim_status for claim-specific information.


Examples:

User:
"What is my deductible?"

Assistant:
"Please provide your member ID."

User:
"Why was my claim denied?"

Assistant:
"Please provide your claim ID."
"""

In [25]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt
)

# Test Agent

In [34]:
response = agent.invoke(
    {
        "messages": [
            (
                "user",
                "How can I contact support?"
            )
        ]
    }
)

# print(response["messages"][-1].content)
print("\nAssistant:", response["messages"][-1].content[1]["text"])
print("input_tokens", response["messages"][-1].usage_metadata["input_tokens"])
print("output_tokens", response["messages"][-1].usage_metadata["output_tokens"])
print("total_tokens", response["messages"][-1].usage_metadata["total_tokens"])


Assistant: 
I can provide you with the general support contact information. You can reach our support team via phone at 1-800-123-4567 or via email at support@healthsecure.com. If you are a member, you can also access support through your member portal. If you need specific member-related support, please provide your member ID.
input_tokens 619
output_tokens 116
total_tokens 735


# Test Agent - CLI

In [ ]:
# test agent with memory - CLI
messages = []

while True:
    user_input = input("\nUser: ")
    
    if user_input.lower() in ["exit", "quit"]:
        break

    messages.append(
        (
            "user",
            user_input
        )
    )

    response = agent.invoke(
        {
            "messages": messages
        }
    )

# print(response["messages"][-1].content)
print("\nAssistant:", response["messages"][-1].content[1]["text"])
print("input_tokens", response["messages"][-1].usage_metadata["input_tokens"])
print("output_tokens", response["messages"][-1].usage_metadata["output_tokens"])
print("total_tokens", response["messages"][-1].usage_metadata["total_tokens"])


User:  How do I become a member?
